In [19]:
def build_vocab():
    """
    Builds a character -> integer lookup table.
    Returns:
        char2idx : dict  — maps character → integer index
        idx2char : dict  — maps integer index → character
        vocab (dict): A dictionary mapping characters to unique integers.
    """
    # All printable characters we need
    charset = tuple(chr(i) for i in range(32, 126))

    special_tokens = ['<PAD>', '<BOS>', '<EOS>', '<UNK>']

    char2idx = {ch: idx for idx, ch in enumerate(special_tokens)}
    for i, ch in enumerate(charset):
        char2idx[ch] = i + len(special_tokens)

    idx2char = {idx: ch for ch, idx in char2idx.items()}

    return char2idx, idx2char
# Test
char2idx, idx2char = build_vocab()
print(f"Vocab size: {len(char2idx)}")
print(f"<PAD>: {char2idx['<PAD>']}")
print(f"<BOS>: {char2idx['<BOS>']}")
print(f"<EOS>: {char2idx['<EOS>']}")
word = "Hey claude"
tokens = [char2idx['<BOS>']] + [char2idx[c] for c in word] + [char2idx['<EOS>']]
print(f"'{word}' → {tokens}")

Vocab size: 98
<PAD>: 0
<BOS>: 1
<EOS>: 2
'Hey claude' → [1, 44, 73, 93, 4, 71, 80, 69, 89, 72, 73, 2]


In [20]:
import torch

def encode_text(text, char2idx, max_len=256):
    """
    Converts a text string into a padded integer tensor.

    Args:
        text     (str)  : input string e.g. "Hello"
        char2idx (dict) : character → index lookup table
        max_len  (int)  : fixed sequence length including BOS and EOS

    Returns:
        tensor (torch.LongTensor): shape (max_len,)
    """
    # Tokenize — use <UNK> for any character not in vocab
    tokens = [char2idx.get(c, char2idx['<UNK>']) for c in text]

    # Truncate to leave room for BOS and EOS
    tokens = tokens[:max_len - 2]

    # Wrap with BOS and EOS
    tokens = [char2idx['<BOS>']] + tokens + [char2idx['<EOS>']]

    # Pad to max_len
    pad_len = max_len - len(tokens)
    tokens = tokens + [char2idx['<PAD>']] * pad_len

    return torch.tensor(tokens, dtype=torch.long)


# Test
char2idx, idx2char = build_vocab()

t1 = encode_text("Hey claude", char2idx, max_len=15)
t2 = encode_text("Hi", char2idx, max_len=15)
t3 = encode_text("This is a very long sentence that exceeds max len", char2idx, max_len=15)

print(t1)
print(t2)
print(t3)
print(f"All shapes: {t1.shape}, {t2.shape}, {t3.shape}")

tensor([ 1, 44, 73, 93,  4, 71, 80, 69, 89, 72, 73,  2,  0,  0,  0])
tensor([ 1, 44, 77,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])
tensor([ 1, 56, 76, 77, 87,  4, 77, 87,  4, 69,  4, 90, 73, 86,  2])
All shapes: torch.Size([15]), torch.Size([15]), torch.Size([15])


In [21]:
from torch import embedding
import torch.nn as nn

embedding = nn.Embedding(num_embeddings=98, embedding_dim=256)
t2 = encode_text("Hi", char2idx, max_len=256)
print(f"Input token IDs: {t2}")
output = embedding(t2)
print(f"Output shape: {output.shape}")

Input token IDs: tensor([ 1, 44, 77,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,

In [22]:
import torch
import torch.nn as nn

class Text_Encoder(nn.Module):
    """
    Encodes a tokenized text sequence into a sequence of embeddings.

    Args:
        vocab_size (int): number of characters in the vocabulary
        embed_dim  (int): dimensionality of the output embeddings
        pad_idx   (int): index used for padding tokens (default: 0)
    Returns:
        tensor (torch.FloatTensor): shape (B, embed_dim)    
    """
    def __init__(self, vocab_size, embed_dim=256, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, 
            embedding_dim=embed_dim,
            padding_idx = pad_idx
        )
    
    def forward(self, x):
        # token_ids shape: (B, max_len)
        token_ids = x
        # Embed each token
        embedded = self.embedding(token_ids) # (B, max_len, embed_dim)

        # Build mask to ignore padding tokens
        mask = (token_ids != self.embedding.padding_idx) # (B, max_len)

        # zero out embeddings at padding positions
        mask_expanded = mask.unsqueeze(-1).float()
        masked = embedded * mask_expanded

        # Average of real token embeddings (ignore padding)
        counts = mask.sum(dim=1, keepdim=True).float() # (B, 1)
        average = masked.sum(dim=1) / counts  # (B, embed_dim)

        return average
# Test
char2idx, idx2char = build_vocab()
vocab_size = len(char2idx)
encoder = Text_Encoder(vocab_size=vocab_size, embed_dim=256, pad_idx=0)

# Simulate a batch of 4 tokenized sentences
texts = ["Hey claude", "Hi", "Good morning", "Write this in my style"]
tokens = torch.stack([encode_text(t, char2idx, max_len=256) for t in texts])

print(f"Input shape:  {tokens.shape}")     # (4, 256)
output = encoder(tokens)
print(f"Output shape: {output.shape}")     # (4, 256)

Input shape:  torch.Size([4, 256])
Output shape: torch.Size([4, 256])


In [ ]:
import math

class Learned_Positional_Encoding(nn.Module):
    """
    Learns a vector for each position during training.
    Args:
        max_len   (int) : maximum sequence length
        embed_dim (int) : embedding dimensionality
    """
    def __init__(self, max_len = 256, embed_dim = 256):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        # x shape: (B, max_len, embed_dim)
        B, T, _ = x.shape
        positions = torch.arrange(T, device=x.device)
        pos_embed = self.pos_embedding(positions) # (T, embed_dim)
        return x + pos_embed.unsqueeze(0) # (B, T, embed_dim)
    
class Sinusoidal_Positional_Encoding(nn.Module):
    """
    Fixed sin/cos positional encoding. No learnable parameters.
    Args:
        max_len   (int) : maximum sequence length
        embed_dim (int) : embedding dimensionality
    """
    def __init__(self, max_len = 256, embed_dim = 256):
        super().__init__()            
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2) * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) # (1, max_len, embed_dim)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :] # (B, T, embed_dim)
# Test
embedding = nn.Embedding(len(char2idx), 256, padding_idx=0)
tokens = torch.stack([encode_text(t, char2idx, max_len = 256) for t in texts]) # (4, 256)
embedded = embedding(tokens) # (4, 256, 256)
print(f"Before Positional Encoding: {embedded.shape}")

learned_pe = Learned_Positional_Encoding(max_len=256, embed_dim=256)
sinu_pe = Sinusoidal_Positional_Encoding(max_len=256, embed_dim=256)

out_learned = learned_pe(embedded)
out_sinu = sinu_pe(embedded)
print(f"After Learned PE: {out_learned.shape}")
print(f"After Sinusoidal PE: {out_sinu.shape}")
print(f"Same output? {torch.allclose(out_learned, out_sinu)}")